In [38]:
import sys
import logging
from pathlib import Path
from typing import List, Mapping, Sequence, Union, Dict
import pandas as pd
import os

import pandas as pd
import dycomutils as common_utils

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

os.chdir(repo_root)
print(repo_root)
sys.path.append(str(repo_root))

from src.utils.graph_manager import GraphManager
from src.utils.utils import load_config, format_dataframe_to_string, regex_add_strings
from src.config.experiment import ExperimentConfig
from src.experiment.ground_truth import GT, SPARQLTemplate, GTAnswer
from src.experiment.test_questions_templates import build_gt_from_template

CONFIG_PATH = "evaluations/biomni-base/config.yaml"
logging.info(f"Loading config: {CONFIG_PATH}")
lconfig = load_config(CONFIG_PATH)
config = ExperimentConfig.model_validate(lconfig)
config


/home/desild/work/research/LLM-Workflow-Explorer


ExperimentConfig(application=ApplicationInfo(description='The Biomni application is an LLM-driven biomedical agent that plans, reasons,\nexecutes generated code, and optionally critiques intermediate answers to solve\nuser-provided biomedical tasks. The provenance captures the user input, system\nprompt generation, each agent generation iteration, code execution, execution\nobservations, final solution, and optional critic feedback as intermediary\nstates of the agent run.\nAgent stages are represented as provone:Executions of provone:Programs,\nincluding user input recording, system prompt generation, LLM response\ngeneration, generated code execution, and critic review. LLM usage within the\nagent is represented as workflow:Generative_Task activity using\nworkflow:Large_Language_Models. Generated responses, executable code,\nobservations, critic feedback, and final solutions are connected across the\nworkflow using provone:Channel and prov:wasInformedBy relations.\ndcterms:identifier

In [39]:
graph_manager = GraphManager(
    config.ttl,
    config.file_paths.execution_kg_loc,
)

gt_list: List[GT] = []


In [40]:
# Question 1
question1 = """
How many "experiment execution" are there in this?
"""
question_sparql1 = """
SELECT distinct ?obj
WHERE {
     ?obj a provone:Execution .
     ?obj rdfs:label ?lbl .
}
"""


entities1 = graph_manager.query(question_sparql1)
entities1


,obj
0,http://testwebsite/testProgram#id_202605050009...
1,http://testwebsite/testProgram#id_202605050009...
2,http://testwebsite/testProgram#id_202605050009...
3,http://testwebsite/testProgram#id_202605050010...
4,http://testwebsite/testProgram#id_202605050010...
5,http://testwebsite/testProgram#id_202605050010...
6,http://testwebsite/testProgram#id_202605050010...
7,http://testwebsite/testProgram#id_202605050010...
8,http://testwebsite/testProgram#id_202605050010...
9,http://testwebsite/testProgram#id_202605050010...


In [41]:
answer1 = f"The answer to the question is {len(entities1)} unique executions."

gt_list.append(
    GT(
        question=question1,
        answer=answer1,
        count=len(entities1),
        sparql=[
            SPARQLTemplate(
                template=question_sparql1,
                description="This SPARQL query counts the number of unique executions by counting distinct identifiers in the provone:Execution class.",
            )
        ],
        #qtype=["multi", "numeric", "ISP"],
        qtype=["multi", "numeric"],
        entities=entities1.to_dict(orient="records"),
    )
)


In [42]:
# Question 3
question3 = """
In what places do we utilize AI in this workflow?
"""
question_sparql3 = """
SELECT distinct ?obj ?desc
WHERE {
     ?obj a workflow:Generative_Task .
     ?obj dc:description ?desc .
}
"""

entities3 = graph_manager.query(question_sparql3, resolve_curie=True)
entities3


,obj,desc
0,Biomni:Generative_Task-id_20260505001006_475,Step 4: Records how the agent used an LLM duri...
1,Biomni:Generative_Task-id_20260505001006_475,Step 4: Records how the agent used an LLM duri...
2,Biomni:Generative_Task-id_20260505001009_972,Step 5: Records how the agent used an LLM duri...
3,Biomni:Generative_Task-id_20260505001009_972,Step 5: Records how the agent used an LLM duri...
4,Biomni:Generative_Task-id_20260505001016_956,Step 8: Records how the agent used an LLM duri...
5,Biomni:Generative_Task-id_20260505001016_956,Step 8: Records how the agent used an LLM duri...
6,Biomni:Generative_Task-id_20260505001023_688,Step 9: Records how the agent used an LLM duri...
7,Biomni:Generative_Task-id_20260505001023_688,Step 9: Records how the agent used an LLM duri...
8,Biomni:Generative_Task-id_20260505001035_720,Step 10: Records how the agent used an LLM dur...
9,Biomni:Generative_Task-id_20260505001035_720,Step 10: Records the LLM use that produced sel...


In [43]:
answer3 = """
The ChatBS System utilizes AI for the following

{}
""".format("\n\n".join([f"{n+1} => {v}" for n,v in enumerate(entities3['desc'].to_list())]))

print(answer3)

gt_list.append(
    GT(
        question=question3,
        answer=answer3,
        sparql=[SPARQLTemplate(template=question_sparql3, description="")],
        entities=entities3.to_dict(orient="records"),
        #qtype=["multi", "entity", "SI"],
        qtype=["multi", "entity"],
    )
)



The ChatBS System utilizes AI for the following

1 => Step 4: Records how the agent used an LLM during the generation iteration 1 step.

2 => Step 4: Records how the agent used an LLM during this generation iteration.

3 => Step 5: Records how the agent used an LLM during the generation iteration 2 step.

4 => Step 5: Records how the agent used an LLM during this generation iteration.

5 => Step 8: Records how the agent used an LLM during the generation iteration 4 step.

6 => Step 8: Records how the agent used an LLM during this generation iteration.

7 => Step 9: Records how the agent used an LLM during the generation iteration 5 step.

8 => Step 9: Records how the agent used an LLM during this generation iteration.

9 => Step 10: Records how the agent used an LLM during the critic review 1 step.

10 => Step 10: Records the LLM use that produced self-critic feedback for the agent.

11 => Step 11: Records how the agent used an LLM during the generation iteration 6 step.

12 => Step 1

In [44]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [45]:
# Question MULT 2: input ports of the functions


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
what are the input parameters of the function used in "{func_desc}" step in the pipeline?
"""
sparql_template = """
SELECT distinct ?flbl ?lbl
WHERE {
     ?obj dc:description ?desc .
     ?obj rdfs:label ?flbl . 
     ?obj a ?class .
     FILTER(REGEX(?flbl, "{func_desc_term}","i"))

     ?obj provone:hasInPort ?port .
     ?port rdfs:label ?lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input parameters to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Funtion name",
            "lbl": "Function returns"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )


for gt in gts:
    #gt.qtype = ["multi", "literal", "SI"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the input parameters of the function used in "Step 10 - Biomni critic review 1" step in the pipeline?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              1  Step 10 - Biomni critic review 1   
              2  Step 10 - Biomni critic review 1   
              3  Step 10 - Biomni critic review 1   
              4  Step 10 - Biomni critic review 1   
              5  Step 10 - Biomni critic review 1   
              6  Step 10 - Biomni critic review 1   
              7  Step 10 - Biomni critic review 1   
              
                                                               lbl  
              0                 Step 10 - candidate solution input  
              1  Step 10 - candidate_solution input port for cr...  
              2                      Step 10 - critic prompt input  
              3  Step 10 - feedback_prompt input port for criti...

In [46]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [47]:
# Question MULT 3 : output ports of functions


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
what are the output parameters of the function used in "{func_desc}" step in the pipeline?
"""
sparql_template = """
SELECT distinct ?flbl ?lbl
WHERE {
     ?obj dc:description ?desc .
     ?obj rdfs:label ?flbl . 
     ?obj a ?class .
     FILTER(REGEX(?flbl, "{func_desc_term}","i"))

     ?obj provone:hasOutPort ?port .
     ?port rdfs:label ?lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input parameters to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Funtion name",
            "lbl": "Function returns"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "literal", "SI"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the output parameters of the function used in "Step 10 - Biomni critic review 1" step in the pipeline?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              1  Step 10 - Biomni critic review 1   
              2  Step 10 - Biomni critic review 1   
              3  Step 10 - Biomni critic review 1   
              
                                                               lbl  
              0                   Step 10 - critic feedback output  
              1  Step 10 - critic_feedback output port for crit...  
              2                   Step 10 - next agent step output  
              3  Step 10 - next_step output port for critic rev...  
ic| answer: GTAnswer(answer_nlp='
                The input parameters to Step 10 - Biomni critic review 1 function.
            
                Funtion name | Function returns
            Step 10 - Biomni c

In [48]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [49]:
# Question MULT 4 : connected functions


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
what are the other function ("program") connected to the function used in "{func_desc}" through port parameters?
"""
sparql_template = """

SELECT DISTINCT ?fn2 ?fn2lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?obj (provone:hasInPort | provone:hasOutPort) ?port .
    ?port provone:connectsTo ?channel .
    ?portt provone:connectsTo ?channel .
    ?fn2 (provone:hasInPort | provone:hasOutPort) ?portt .
    FILTER(?obj != ?fn2)
    
    ?fn2 rdfs:label ?fn2lbl .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The connected functions to {func_desc} function.

    {answer_inst}
    """
    
    _ent = entities[['fn2', 'fn2lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "fn2": "Funtion URI",
            "fn2lbl": "Function name"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "literal", "entity", "SI"]
    gt.qtype = ["multi",  "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the other function ("program") connected to the function used in "Step 10 - Biomni critic review 1" through port parameters?
                '''
ic| entities:                                                  fn2  \
              0  http://testwebsite/testProgram#generate_agent_...   
              1  http://testwebsite/testProgram#generate_agent_...   
              
                                                      fn2lbl  
              0   Step 9 - Biomni LLM generation iteration 5  
              1  Step 11 - Biomni LLM generation iteration 6  
ic| answer: GTAnswer(answer_nlp='
                The connected functions to Step 10 - Biomni critic review 1 function.
            
                Funtion URI | Function name
            http://testwebsite/testProgram#generate_agent_response_5 | Step 9 - Biomni LLM generation iteration 5
            http://testwebsite/testProgram#generate_agent_response_6 | Step 11 - Biomni LLM generation ite

In [50]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [51]:
# Question MULT 5 : number of execution of the functions


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
what are the executions of the {func_desc} step in the pipeline?
"""
sparql_template = """

SELECT DISTINCT ?exe
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The executions of the  {func_desc} function are.

    {answer_inst}
    """
    
    _ent = entities[['exe']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "exe": "Executions"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "entity", "ISP"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                what are the executions of the Step 10 - Biomni critic review 1 step in the pipeline?
                '''
ic| entities:                                                  exe
              0  http://testwebsite/testProgram#id_202605050010...
ic| answer: GTAnswer(answer_nlp='
                The executions of the  Step 10 - Biomni critic review 1 function are.
            
                Executions
            http://testwebsite/testProgram#id_20260505001035_720
                ', entities=[{'exe': 'http://testwebsite/testProgram#id_20260505001035_720'}])
ic| rquestion: '''
                what are the executions of the Step 4 - Biomni LLM generation iteration 1 step in the pipeline?
                '''
ic| entities:                                                  exe
              0  http://testwebsite/testProgram#id_202605050010...
ic| answer: GTAnswer(answer_nlp='
                The executions of the  Step 4 - Biomni LLM generation iteration 1 func

In [52]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [53]:
# Question MULT 6 : who was the agent that ran the functions 


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
who were the agents that ran the {func_desc} step in the pipeline?
"""
sparql_template = """

SELECT DISTINCT ?agent ?albl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl . 
    ?obj a ?class .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?asso prov:agent ?agent .
    ?agent rdfs:label ?albl
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The executions of the  {func_desc} function are.

    {answer_inst}
    """
    
    _ent = entities[['albl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "albl": "Agents"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "entity", "literal", "ISP"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                who were the agents that ran the Step 10 - Biomni critic review 1 step in the pipeline?
                '''
ic| entities:                                        agent  \
              0   http://testwebsite/testProgram#test_user   
              1   http://testwebsite/testProgram#test_user   
              2   http://testwebsite/testProgram#test_user   
              3   http://testwebsite/testProgram#test_user   
              4   http://testwebsite/testProgram#test_user   
              5   http://testwebsite/testProgram#test_user   
              6   http://testwebsite/testProgram#test_user   
              7   http://testwebsite/testProgram#test_user   
              8   http://testwebsite/testProgram#test_user   
              9   http://testwebsite/testProgram#test_user   
              10  http://testwebsite/testProgram#test_user   
              11  http://testwebsite/testProgram#test_user   
              
                                    

In [54]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [55]:
# Question MULT 7 : Is the function made by a Generative Task  


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
Is the function with a description as "{func_desc}" made by using Large Language Model output?
"""

sparql_template = """
SELECT DISTINCT ?fnlbl ?llm_gen_out
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?fnlbl .
    FILTER(REGEX(?fnlbl, "{func_desc_term}", "i"))

    OPTIONAL {
        ?llm_out sio:SIO_000202 ?obj .
        ?llm_out sio:SIO_000232 ?llm .
        ?llm_gen prov:used ?llm .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The {func_desc} function {decision} by a Generative Task.

    {answer_inst}
    """
    
    _ent = entities[['fnlbl', 'llm_gen_out']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "obj": "Function",
            'llm_gen_out': "Generative Task used to generate function"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision = "is made" if _ent['llm_gen_out'].tolist()[0] != "None" else "is not made"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["single", "bool", "SI"]
    gt.qtype = ["single", "bool"]
    gt.decision = gt.entities[0]['llm_gen_out'] != "None"
gt_list.extend(gts)

ic| rquestion: '''
                Is the function with a description as "Step 10 - Biomni critic review 1" made by using Large Language Model output?
                '''
ic| entities:                               fnlbl llm_gen_out
              0  Step 10 - Biomni critic review 1        None
ic| answer: GTAnswer(answer_nlp='
                The Step 10 - Biomni critic review 1 function is not made by a Generative Task.
            
                fnlbl | Generative Task used to generate function
            Step 10 - Biomni critic review 1 | None
                ', entities=[{'fnlbl': 'Step 10 - Biomni critic review 1', 'llm_gen_out': 'None'}])
ic| rquestion: '''
                Is the function with a description as "Step 4 - Biomni LLM generation iteration 1" made by using Large Language Model output?
                '''
ic| entities:                                         fnlbl llm_gen_out
              0  Step 4 - Biomni LLM generation iteration 1        None
ic| answer: GTAnswe

In [56]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [57]:
# Question MULT 8 : Does any of the executions of the function contain a Generative Task  


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
Does any of the executions of the "{func_desc}" function made has a Generative Task as a component within?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?llm_gen_out
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    OPTIONAL {
        ?exe sio:SIO_000369 ?llm_gen .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    Executions of the {func_desc} function {decision} a Generative Task that utilize an LLM.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'llm_gen_out']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'llm_gen_out': "Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision = "contains" if _ent['llm_gen_out'].tolist()[0] != "None" else "does not contain"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["single", "bool", "ISP"]
    gt.qtype = ["single", "bool"]
    gt.decision = gt.entities[0]['llm_gen_out'] != "None"
gt_list.extend(gts)

ic| rquestion: '''
                Does any of the executions of the "Step 10 - Biomni critic review 1" function made has a Generative Task as a component within?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              
                                                               exe  \
              0  http://testwebsite/testProgram#id_202605050010...   
              
                                                       llm_gen_out  
              0  http://testwebsite/testProgram#Generative_Task...  
ic| answer: GTAnswer(answer_nlp='
                Executions of the Step 10 - Biomni critic review 1 function contains a Generative Task that utilize an LLM.
            
                Function | Execution | Generative Task used within execution
            Step 10 - Biomni critic review 1 | http://testwebsite/testProgram#id_20260505001035_720 | http://testwebsite/testProgram#Generative_Task-id_202

In [58]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [59]:
# Question MULT 9 : Does any of the executions of the function contain a Generative Task within and the function itself is generated by an LLM


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
Does any of the executions of the {func_desc} function made has a Generative Task within as a component and also {func_desc} 
function itself is generated by a Generative Task?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?llm_gen_out ?llm_gen_out2
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    OPTIONAL {
        ?exe sio:SIO_000369 ?llm_gen .
    }

    BIND(COALESCE(?llm_gen, "None") AS ?llm_gen_out)
    
    OPTIONAL {
        ?llm_out2 sio:SIO_000202 ?obj .
        ?llm_out2 sio:SIO_000232 ?llm2 .
        ?llm_gen2 prov:used ?llm2 .
    }

    BIND(COALESCE(?llm_gen2, "None") AS ?llm_gen_out2)
}

"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    1. Executions of the {func_desc} function {decision1} a Generative Task that utilize an LLM.
    2. The {func_desc} function {decision2} by a Generative Task.

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'llm_gen_out', 'llm_gen_out2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'llm_gen_out': "Generative Task used within execution",
            'llm_gen_out2': "Generative Task used to generate function"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent,
        decision1 = "contains" if _ent['llm_gen_out'].tolist()[0] != "None" else "does not contain",
        decision2 = "is generated" if _ent['llm_gen_out2'].tolist()[0] != "None" else "is not generated"
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["single", "bool", "ISP"]
    gt.qtype = ["single", "bool"]
    gt.decision = (gt.entities[0]['llm_gen_out'] != "None") and (gt.entities[0]['llm_gen_out2'] != "None")

gt_list.extend(gts)

ic| rquestion: '''
                Does any of the executions of the Step 10 - Biomni critic review 1 function made has a Generative Task within as a component and also Step 10 - Biomni critic review 1 
                function itself is generated by a Generative Task?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              
                                                               exe  \
              0  http://testwebsite/testProgram#id_202605050010...   
              
                                                       llm_gen_out llm_gen_out2  
              0  http://testwebsite/testProgram#Generative_Task...         None  
ic| answer: GTAnswer(answer_nlp='
                1. Executions of the Step 10 - Biomni critic review 1 function contains a Generative Task that utilize an LLM.
                2. The Step 10 - Biomni critic review 1 function is not generated by a Generative Task.
     

In [60]:
question_sparql1 = """
SELECT distinct ?obj ?lbl ?llm_gen
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
     
     ?asso prov:hadPlan ?obj .
     ?exe prov:qualifiedAssociation ?asso .
     
     ?exe sio:SIO_000369 ?llm_gen .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [61]:
# Question MULT 10 : Does any of the executions of the function contain a Generative Task within and the function itself is generated by an LLM

template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
What are the inputs of the Generative Task that was a component in the executions of  the {func_desc} function?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    ?exe sio:SIO_000369 ?llm_gen .
    ?llm_gen prov:used ?llm .
    ?llm sio:SIO_000230 ?inp .
    ?inp prov:value ?lbl . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the LLM used for the Generative Task in the executions of the {func_desc} function

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'lbl': "Inputs for the Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent[['exe', 'lbl']].to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "entity", 'label', "SI"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                What are the inputs of the Generative Task that was a component in the executions of  the Step 10 - Biomni critic review 1 function?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              1  Step 10 - Biomni critic review 1   
              2  Step 10 - Biomni critic review 1   
              3  Step 10 - Biomni critic review 1   
              
                                                               exe  \
              0  http://testwebsite/testProgram#id_202605050010...   
              1  http://testwebsite/testProgram#id_202605050010...   
              2  http://testwebsite/testProgram#id_202605050010...   
              3  http://testwebsite/testProgram#id_202605050010...   
              
                                                               lbl  
              0  I first imported the ADMET prediction function...  
              1  
           

In [62]:
# Question MULT 11 :


template_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
sparql_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

question_template = """
What are the outputs of the Generative Task that was a component in the executions of the {func_desc} function?
"""

sparql_template = """
SELECT DISTINCT ?flbl ?exe ?lbl
WHERE {
    ?obj dc:description ?desc .
    ?obj rdfs:label ?flbl .
    FILTER(REGEX(?flbl, "{func_desc_term}","i"))

    ?asso prov:hadPlan ?obj .
    ?exe prov:qualifiedAssociation ?asso .

    ?exe sio:SIO_000369 ?llm_gen .
    ?llm_gen prov:used ?llm .
    ?llm sio:SIO_000229 ?out .
    ?out sio:SIO_000202 ?data . 
    ?data prov:value ?lbl . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the LLM used for the Generative Task in the executions of the {func_desc} function

    {answer_inst}
    """
    
    _ent = entities[['flbl', 'exe', 'lbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            "flbl": "Function",
            'exe': "Execution",
            'lbl': "Inputs for the Generative Task used within execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        func_desc = question_specs["func_desc"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "entity", 'label', "ISP"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                What are the outputs of the Generative Task that was a component in the executions of the Step 10 - Biomni critic review 1 function?
                '''
ic| entities:                                flbl  \
              0  Step 10 - Biomni critic review 1   
              1  Step 10 - Biomni critic review 1   
              
                                                               exe  \
              0  http://testwebsite/testProgram#id_202605050010...   
              1  http://testwebsite/testProgram#id_202605050010...   
              
                                                               lbl  
              0                                           generate  
              1  **Critical Review of the Previous Response**
              ...  
ic| answer: GTAnswer(answer_nlp='
                The inputs to the LLM used for the Generative Task in the executions of the Step 10 - Biomni critic review 1 function
            
            

In [63]:
# Question MULT 11 :

_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?flbl, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        template_spec.append({
            "exe_id":obj['exe']
        })
        
        sparql_spec.append({
            "exe_id":obj['exe']
        })

question_template = """
What execution created the output data that was used as an input for the generative task within the execution "{exe_id}"?
"""

sparql_template = """
SELECT DISTINCT ?exe2
    WHERE {
        <{exe_id}> sio:SIO_000369 ?llm_gen .
        ?llm_gen prov:used ?llm .
        ?llm sio:SIO_000229 ?out .
        ?out sio:SIO_000202 ?data . 
        ?data prov:wasGeneratedBy ?exe2 . 
}
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The inputs to the Generative Task in the execution of "{exe_id}" was generated by executions

    {answer_inst}
    """
    
    _ent = entities[['exe2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'exe2': "Execution"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "entity", 'label', "ISP"]
    gt.qtype = ["multi", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                What execution created the output data that was used as an input for the generative task within the execution "http://testwebsite/testProgram#id_20260505001035_720"?
                '''
ic| entities:                                                 exe2
              0  http://testwebsite/testProgram#id_202605050010...
              1  http://testwebsite/testProgram#id_202605050010...
ic| answer: GTAnswer(answer_nlp='
                The inputs to the Generative Task in the execution of "http://testwebsite/testProgram#id_20260505001035_720" was generated by executions
            
                Execution
            http://testwebsite/testProgram#id_20260505001035_720
            http://testwebsite/testProgram#id_20260505001037_336
                ', entities=[{'exe2': 'http://testwebsite/testProgram#id_20260505001035_720'}, {'exe2': 'http://testwebsite/testProgram#id_20260505001037_336'}])
ic| rquestion: '''
                What execution created th

In [64]:
question_sparql1 = """
SELECT distinct ?obj ?lbl ?llm_gen
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
     
     ?asso prov:hadPlan ?obj .
     ?exe prov:qualifiedAssociation ?asso .
     
     ?exe sio:SIO_000369 ?llm_gen .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [65]:
# Question MULT 12 :

_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?flbl, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        template_spec.append({
            "exe_id":obj['exe']
        })
        
        sparql_spec.append({
            "exe_id":obj['exe']
        })

question_template = """
The execution "{exe_id}" used data were created by what Generative Task?
"""

sparql_template = """
SELECT DISTINCT ?gtask ?glbl
            WHERE {
                <{exe_id}> prov:used ?data .
                ?llm_out sio:SIO_000202 ?data .
                ?llm_out sio:SIO_000232 ?llm .
                ?gtask prov:used ?llm .
                ?gtask rdfs:label ?glbl
        }
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The Generative Task that generated the data that was used by the execution "{exe_id}" is 

    {answer_inst}
    """
    
    _ent = entities[['gtask', 'glbl']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'gtask': "Generative Task"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent[['gtask']].to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["single", "entity", "ISP"]
    gt.qtype = ["single", "entity"]
gt_list.extend(gts)

ic| rquestion: '''
                The execution "http://testwebsite/testProgram#id_20260505001035_720" used data were created by what Generative Task?
                '''
ic| entities:                                                gtask  \
              0  http://testwebsite/testProgram#Generative_Task...   
              1  http://testwebsite/testProgram#Generative_Task...   
              
                                                              glbl  
              0  Step 9 - Biomni LLM use for generation iterati...  
              1    Step 9 - generation iteration 5 generative task  
ic| answer: GTAnswer(answer_nlp='
                The Generative Task that generated the data that was used by the execution "http://testwebsite/testProgram#id_20260505001035_720" is 
            
                Generative Task | glbl
            http://testwebsite/testProgram#Generative_Task-id_20260505001023_688 | Step 9 - Biomni LLM use for generation iteration 5
            http://testweb

In [66]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [67]:
# Input port data created by AI 

_spec = template_spec[:5] if len(template_spec) > 5 else template_spec

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?flbl, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        usage_query = """
        SELECT DISTINCT ?usage ?lbl
        WHERE {
            <{exe_id}> prov:qualifiedUsage ?usage .
            ?usage provone:hadInPort ?port .
            ?port rdfs:label ?lbl .
            }
        """
        usages = graph_manager.query(
                regex_add_strings(
                    usage_query,
                    exe_id = obj["exe"]
                )
            )
        
        for usage in usages.to_dict("records"):
            # df = graph_manager.query(
            #     regex_add_strings(
            #         sparql_template,
            #         exe_id = obj["exe"],
            #         port_label = usage["lbl"].split("@")[0]
            #     )
            # )

            # print(df)
            
            template_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
            sparql_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
question_template = """
Was the input to the "{port_label}" in the execution "{exe_id}" generated by a Generative Task?
"""

sparql_template = """
SELECT DISTINCT ?plbl ?gtask2
            WHERE {
                <{exe_id}> prov:qualifiedUsage ?usage .
                ?usage provone:hadInPort ?port .
                ?port rdfs:label ?plbl .
                FILTER(REGEX(?plbl, "{port_label}","i"))
                
                ?usage provone:hadEntity ?data .
                OPTIONAL {
                    ?llm_out sio:SIO_000202 ?data .
                    ?llm_out sio:SIO_000232 ?llm .
                    ?gtask prov:used ?llm .
                }
                
                BIND(COALESCE(?gtask, "None") AS ?gtask2)
        }
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The input to the "{port_label}" in the execution "{exe_id}" {decision} generated by a 
    Generative Task

    {answer_inst}
    """
    
    _ent = entities[['gtask2']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'gtask2': "Generative Task"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        port_label = question_specs["port_label"],
        decision = "was" if _ent['gtask2'].tolist()[0] != "None" else "was not",
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["single", "bool", "ISP"]
    gt.qtype = ["single", "bool"]
    gt.decision = gt.entities[0]['gtask2'] != "None"
gt_list.extend(gts)

ic| rquestion: '''
                Was the input to the "Step 10 - critic prompt input" in the execution "http://testwebsite/testProgram#id_20260505001035_720" generated by a Generative Task?
                '''
ic| entities:                             plbl gtask2
              0  Step 10 - critic prompt input   None
ic| answer: GTAnswer(answer_nlp='
                The input to the "Step 10 - critic prompt input" in the execution "http://testwebsite/testProgram#id_20260505001035_720" was not generated by a 
                Generative Task
            
                Generative Task
            None
                ', entities=[{'gtask2': 'None'}])
ic| rquestion: '''
                Was the input to the "Step 10 - feedback_prompt input port for critic review 1" in the execution "http://testwebsite/testProgram#id_20260505001035_720" generated by a Generative Task?
                '''
ic| entities:                                                 plbl gtask2
              0  Step 10 - f

In [68]:
question_sparql1 = """
SELECT distinct ?obj ?lbl
WHERE {
     ?obj a provone:Program .
     ?obj rdfs:label ?lbl .
}
"""


programs = graph_manager.query(question_sparql1).to_dict(orient="records")
template_spec = [{"func_desc":x["lbl"] , "func_desc_term":x["lbl"]} for x in programs]
template_spec;

In [69]:
# Output data of execution with port

_spec = template_spec[:5] if len(template_spec) > 5 else template_spec
# sparql_template = """
# SELECT DISTINCT ?data
#             WHERE {
#                 <{exe_id}> prov:qualifiedGeneration ?generation .
#                 ?generation provone:hadOutPort ?port .
#                 ?port rdfs:label ?plbl .
#                 FILTER(REGEX(?plbl, "{port_label}","i"))
                
#                 ?generation provone:hadEntity ?data .
#         }
# """

template_spec = []
sparql_spec = []

for _s in _spec:
    exe_query = """
    SELECT DISTINCT ?exe
    WHERE {
        ?obj dc:description ?desc .
        ?obj rdfs:label ?flbl .
        FILTER(REGEX(?flbl, "{func_desc_term}","i"))
        
        ?asso prov:hadPlan ?obj .
        ?exe prov:qualifiedAssociation ?asso .      
        
        }
    """
    executions = graph_manager.query(
        regex_add_strings(exe_query, func_desc_term = _s["func_desc_term"])
    )


    for obj in executions.to_dict("records"):
        usage_query = """
        SELECT DISTINCT ?lbl
        WHERE {
            <{exe_id}> prov:qualifiedGeneration ?generation .
            ?generation provone:hadOutPort ?port .
            ?port rdfs:label ?lbl .
            }
        """
        usages = graph_manager.query(
                regex_add_strings(
                    usage_query,
                    exe_id = obj["exe"]
                )
            )
        
        for usage in usages.to_dict("records"):
            # df = graph_manager.query(
            #     regex_add_strings(
            #         sparql_template,
            #         exe_id = obj["exe"],
            #         port_label = usage["lbl"].split("@")[0]
            #     )
            # )

            # print(df)
            
            template_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })
            
            sparql_spec.append({
                "exe_id":obj['exe'],
                "port_label" : usage["lbl"].split("@")[0]
            })

           
question_template = """
what is the number of output for the "{port_label}" in the execution "{exe_id}" ?
"""

sparql_template = """
SELECT DISTINCT ?data
            WHERE {
                <{exe_id}> prov:qualifiedGeneration ?generation .
                ?generation provone:hadOutPort ?port .
                ?port rdfs:label ?plbl .
                FILTER(REGEX(?plbl, "{port_label}","i"))
                
                ?generation provone:hadEntity ?data .
        }
"""

def answer_template_function(
    entities:pd.DataFrame, 
    question_specs:Mapping[str, Union[str,int]], 
    sparql_specs:Mapping[str, Union[str,int]]
    ) -> GTAnswer:

    answer_template = """
    The number of output for the "{port_label}" in the execution "{exe_id}" is {count}

    {answer_inst}
    """
    
    _ent = entities[['data']]
    _temp_ent = format_dataframe_to_string(
        _ent,
        header_map={
            'data': "Output Data"
        }
    )
    
    answer_nlp = regex_add_strings(
        answer_template, 
        exe_id = question_specs["exe_id"],
        port_label = question_specs["port_label"],
        count = len(_ent['data'].tolist()),
        answer_inst = _temp_ent
        )
    
    return GTAnswer(
        answer_nlp= answer_nlp,
        entities= _ent.to_dict(orient="records")
    )

gts = build_gt_from_template(
        template=question_template,
        answer_template=answer_template_function,
        sparql_query=sparql_template,
        specs_template=template_spec,
        specs_sparql=sparql_spec,
        graph_manager= graph_manager,
        verbose=True
    )

for gt in gts:
    #gt.qtype = ["multi", "numeric", "ISP"]
    gt.qtype = ["multi", "numeric"]
    gt.count = len(gt.entities)
gt_list.extend(gts)

ic| rquestion: '''
                what is the number of output for the "Step 10 - next agent step output" in the execution "http://testwebsite/testProgram#id_20260505001035_720" ?
                '''
ic| entities:                                                 data
              0  http://testwebsite/testProgram#Data-id_2026050...
ic| answer: GTAnswer(answer_nlp='
                The number of output for the "Step 10 - next agent step output" in the execution "http://testwebsite/testProgram#id_20260505001035_720" is 1
            
                Output Data
            http://testwebsite/testProgram#Data-id_20260505001035_545-next_step
                ', entities=[{'data': 'http://testwebsite/testProgram#Data-id_20260505001035_545-next_step'}])
ic| rquestion: '''
                what is the number of output for the "Step 10 - next_step output port for critic review 1" in the execution "http://testwebsite/testProgram#id_20260505001035_720" ?
                '''
ic| entities:         

In [70]:
os.makedirs(os.path.dirname(config.gt.save_loc), exist_ok=True)
for i, gt in enumerate(gt_list):
    gt.id = f"gt_{i}"

common_utils.serialization.save_jsonl(
    [gt.model_dump() for gt in gt_list],
    config.gt.save_loc
)

In [71]:
gt_list

[GT(id='gt_0', question='\nHow many "experiment execution" are there in this?\n', answer='The answer to the question is 12 unique executions.', entities=[{'obj': 'http://testwebsite/testProgram#id_20260505000949_750'}, {'obj': 'http://testwebsite/testProgram#id_20260505000949_550'}, {'obj': 'http://testwebsite/testProgram#id_20260505000949_654'}, {'obj': 'http://testwebsite/testProgram#id_20260505001013_746'}, {'obj': 'http://testwebsite/testProgram#id_20260505001035_720'}, {'obj': 'http://testwebsite/testProgram#id_20260505001042_313'}, {'obj': 'http://testwebsite/testProgram#id_20260505001006_475'}, {'obj': 'http://testwebsite/testProgram#id_20260505001009_972'}, {'obj': 'http://testwebsite/testProgram#id_20260505001011_890'}, {'obj': 'http://testwebsite/testProgram#id_20260505001016_956'}, {'obj': 'http://testwebsite/testProgram#id_20260505001023_688'}, {'obj': 'http://testwebsite/testProgram#id_20260505001037_336'}], sparql=[SPARQLTemplate(template='\nSELECT distinct ?obj\nWHERE {\

In [72]:
all_qtypes = {}
for gt in gt_list:
    if gt.qtype:
        for t in gt.qtype:
            if t in all_qtypes:
                all_qtypes[t] += 1
            else:
                all_qtypes[t] = 1
        
all_qtypes

{'multi': 62, 'numeric': 21, 'entity': 46, 'single': 44, 'bool': 39}

In [73]:
all_qtypes = {}
for gt in gt_list:
    if gt.qtype:
        tags_concat = "|".join(sorted(gt.qtype))
        if tags_concat not in all_qtypes:
            all_qtypes[tags_concat] = 0
        
        all_qtypes[tags_concat] += 1
        
all_qtypes

{'multi|numeric': 21,
 'entity|multi': 41,
 'bool|single': 39,
 'entity|single': 5}

In [74]:
sel_groups = ["multi", "single|bool", "single|entity"]
groups = {}
_temp = []
for gt in gt_list:
    if gt.qtype:
        nadd = False
        for sg in sel_groups:
            tags = sg.split("|")
            if len(set(tags) - set(gt.qtype)) == 0:
                nadd = True
                
        if not nadd:
            _temp.append(gt)
                
        
for gt in _temp:
    if gt.qtype:
        
        _g = "|"+"|".join(sorted(gt.qtype))+"|"
        if _g not in groups:
            groups[_g] = 0
        
        groups[_g] += 1
        
groups
        

{}